In [1]:
!pip install clustering-benchmarks -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 27.0 MB/s eta 0:00:00


In [2]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [3]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

## Function get_scores

Get the NCA score of a specific dataset using genie mst algorithm

In [4]:
import genieclust
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Function get_scores_with_tsne

Get the NCA score of a specific dataset applying the tsne transformation to the data and then using genie mst algorithm

In [5]:
import genieclust
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_tsne(battery, dataset, apply_scale=False, pca_components=-1, tsne_components=2, tsne_perplexity=30):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  if pca_components != -1:
    pca = PCA(n_components=pca_components)
    X_transformed = pca.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  tsne = TSNE(n_components=tsne_components, perplexity=tsne_perplexity, random_state=42) # Reduce to 2 dimensions
  X_transformed = tsne.fit_transform(X_transformed)
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Function get_scores_with_umap

Get the NCA score of a specific dataset applying the umap transformation to the data and then using genie mst algorithm

In [6]:
import genieclust
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_umap(battery, dataset, apply_scale=False, pca_components=-1, umap_components=2, n_neighbors=15, min_dist=0.1):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  if pca_components != -1:
    pca = PCA(n_components=pca_components)
    X_transformed = pca.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  umap = UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist, random_state=42, n_jobs=1) # Reduce to 2 dimensions
  X_transformed = umap.fit_transform(X_transformed)
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Execute get_scores on all datasets as baseline

In [7]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 2/2 [00:25<00:00, 12.74s/it]


In [8]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,1.000000
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,1.000000
7,fcps,twodiamonds,0.987500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.435664


## Execute get_scores_with_tsne on all datasets trying different values for perplexity

In [9]:
tsne_perplexity = [15,30,45]

scores_lists = {}
for perplexity in tqdm.tqdm(tsne_perplexity, desc="Processing Datasets"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+TSNE'+ str(perplexity) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_tsne(battery, dataset, tsne_perplexity=50))

df_tsne = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 3/3 [07:55<00:00, 158.44s/it]


In [10]:
df_tsne

,Genie+TSNE15 NCA Score,Genie+TSNE30 NCA Score,Genie+TSNE45 NCA Score
0,1.000000,1.000000,1.000000
1,1.000000,1.000000,1.000000
2,0.522972,0.335121,0.522972
3,1.000000,1.000000,1.000000
4,1.000000,1.000000,1.000000
5,1.000000,1.000000,1.000000
6,1.000000,1.000000,1.000000
7,1.000000,1.000000,1.000000
8,1.000000,1.000000,1.000000
9,0.409690,0.305295,0.312438


## Execute get_scores_with_umap on all datasets trying different values for n_neighbors

In [11]:
umap_neighbors = [2,10,20,50]

scores_lists = {}
for neighbors in tqdm.tqdm(umap_neighbors, desc="Processing Datasets"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+UMAP'+ str(neighbors) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_umap(battery, dataset, n_neighbors=neighbors))

df_umap = pd.DataFrame.from_dict(scores_lists)

Processing Datasets:   0%|          | 0/4 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
Processing Datasets: 100%|██████████| 4/4 [03:32<00:00, 53.20s/it]


In [12]:
df_umap

,Genie+UMAP2 NCA Score,Genie+UMAP10 NCA Score,Genie+UMAP20 NCA Score,Genie+UMAP50 NCA Score
0,0.462500,1.000000,1.000000,1.000000
1,0.190000,1.000000,1.000000,1.000000
2,0.253906,0.364185,0.333171,0.930108
3,0.315625,1.000000,1.000000,1.000000
4,0.250000,1.000000,1.000000,1.000000
5,0.691332,1.000000,1.000000,1.000000
6,0.406667,0.996667,1.000000,1.000000
7,0.362500,0.997500,0.997500,1.000000
8,0.480315,1.000000,1.000000,1.000000
9,0.416334,0.346932,0.444670,0.306187


# Final Results

In [15]:
df = pd.concat([df, df_tsne, df_umap], axis=1)
numerical_cols = df.columns[2:]
df.style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+TSNE15 NCA Score,Genie+TSNE30 NCA Score,Genie+TSNE45 NCA Score,Genie+UMAP2 NCA Score,Genie+UMAP10 NCA Score,Genie+UMAP20 NCA Score,Genie+UMAP50 NCA Score
0,fcps,atom,1.000000,1.000000,1.000000,1.000000,0.462500,1.000000,1.000000,1.000000
1,fcps,chainlink,1.000000,1.000000,1.000000,1.000000,0.190000,1.000000,1.000000,1.000000
2,fcps,engytime,0.918870,0.522972,0.335121,0.522972,0.253906,0.364185,0.333171,0.930108
3,fcps,hepta,1.000000,1.000000,1.000000,1.000000,0.315625,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,0.250000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,0.691332,1.000000,1.000000,1.000000
6,fcps,tetra,1.000000,1.000000,1.000000,1.000000,0.406667,0.996667,1.000000,1.000000
7,fcps,twodiamonds,0.987500,1.000000,1.000000,1.000000,0.362500,0.997500,0.997500,1.000000
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,0.480315,1.000000,1.000000,1.000000
9,uci,ecoli,0.435664,0.409690,0.305295,0.312438,0.416334,0.346932,0.444670,0.306187
